# Silver Layer
## Data Preparation

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_silver
""")

In [0]:
# Carregar dados da camada Bronze (ad_inc_raw)
df_raw = spark.table("adult_income.adult_income_raw.ad_inc_raw").toPandas()

# Convert string columns to categorical dtype
for col in ['workclass', 'occupation', 'native-country']:
    df_raw[col] = df_raw[col].astype('category')

# 1) Adicionar a categoria 'Unemployed' apenas se ela ainda não existir
if 'Unemployed' not in df_raw['workclass'].cat.categories:
    df_raw['workclass'] = df_raw['workclass'].cat.add_categories(['Unemployed'])

# 1.1) Preencher os valores nulos
df_raw['workclass'] = df_raw['workclass'].fillna('Unemployed')

# 2) Tratamento de missing values para 'occupation' e 'native-country'
for col in ['occupation', 'native-country']:
    if 'missing' not in df_raw[col].cat.categories:
        df_raw[col] = df_raw[col].cat.add_categories(['missing'])
    df_raw[col] = df_raw[col].fillna('missing')

# Convert pandas DataFrame to Spark DataFrame
df_silver = spark.createDataFrame(df_raw)

# Write the transformed data to the Silver layer
silver_data_path = "adult_income.adult_income_silver.ad_inc_silver"
df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("ad_inc_silver")

print("Silver layer processing completed.")